1. SETUP


In [1]:
#imports
import os
import pdfplumber
import re
import pandas as pd
from tqdm import tqdm 

In [2]:
#dataset path
BASE_DIR = os.path.expanduser('/Users/prabinrimal/Desktop/New>>>>>/Collected Resume Datasets/pdfs')

#output paths
OUTPUT_CSV = os.path.expanduser('./resumes_extracted.csv')
ERROR_CSV = os.path.expanduser('./resume_extraction_errors.csv')

#checking
print(os.path.exists(BASE_DIR))
print(sorted(os.listdir(BASE_DIR))[:20])

True
['.DS_Store', 'AI ML', 'AR VR', 'BI', 'Blockchain ', 'Cloud Solutions Architect', 'Cybersecurity Analyst', 'DESIGNER', 'DIGITAL MEDIA', 'DOT NET ', 'Data Science', 'DevOps ', 'Full Stack Developer', 'Game Developer', 'HR', 'IT Project Manager', 'IT Specialist', 'IoT Engineer', 'MERN Stack Developer', 'Mobile App Developer']


In [3]:
#extraction parameters
MIN_TEXT_LENGTH = 50

2. Extraction Function


In [ ]:
#categories
def normalize_category(folder_name):
    name = folder_name.strip()
    name = re.sub(r'\s+resumes$', '', name, flags=re.IGNORECASE)
    return name.strip()

#text extraction
def _has_merged_tokens(text, max_word_len=25):
    return any(len(w) > max_word_len for w in text.split())


def _rebuild_from_words(page, space_gap_ratio=0.3):
    words = page.extract_words(
        x_tolerance=1.5,
        y_tolerance=3,
        keep_blank_chars=False,
        use_text_flow=False,
    )
    if not words:
        return ""

    lines = {}
    for w in words:
        key = round(w["top"], 1)
        lines.setdefault(key, []).append(w)

    out_lines = []
    for top in sorted(lines.keys()):
        line_words = sorted(lines[top], key=lambda w: w["x0"])
        line_parts = []
        prev_x1 = None
        for w in line_words:
            if prev_x1 is not None:
                gap = w["x0"] - prev_x1
                avg_char_w = (w["x1"] - w["x0"]) / max(len(w["text"]), 1)
                if gap > avg_char_w * space_gap_ratio:
                    line_parts.append(" ")
            line_parts.append(w["text"])
            prev_x1 = w["x1"]
        out_lines.append("".join(line_parts))

    return "\n".join(out_lines)


def extract_text(pdf_path):
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text(
                    x_tolerance=1.5,
                    y_tolerance=3,
                    keep_blank_chars=False,
                )
                if not page_text:
                    continue
                if _has_merged_tokens(page_text):
                    page_text = _rebuild_from_words(page)

                text += page_text + "\n"
    except Exception as e:
        return None, str(e)
    return text, None

#validation of extracted text
def extracted_resume_text(pdf_path):
    text, err = extract_text(pdf_path)
    if text and len(text.strip()) >= MIN_TEXT_LENGTH:
        return text.strip(), None

# Collect files
all_files = []
raw_folders = sorted([d for d in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, d))])
for folder in raw_folders:
    category = normalize_category(folder)
    folder_path = os.path.join(BASE_DIR, folder)
    for fname in os.listdir(folder_path):
        if fname.lower().endswith('.pdf'):
            all_files.append((os.path.join(folder_path, fname), category))

#Reporting
print(f"Found {len(all_files)} PDF files.")
check_df = pd.DataFrame(all_files, columns=['file_path', 'category'])
print(check_df['category'].value_counts())

Found 1603 PDF files.
category
IT Specialist                121
AI ML                        120
HR                           110
DESIGNER                     107
Software Engineer            105
DIGITAL MEDIA                 96
Full Stack Developer          87
Data Science                  85
DevOps                        82
SQL Developer                 82
DOT NET                       69
Testing and QA                68
Robotics Engineer             61
MERN Stack Developer          58
Game Developer                48
Blockchain                    43
Technology Consultant         41
Python Developer              37
Systems Administrator         37
Network Engineer              36
Mobile App Developer          34
Cloud Solutions Architect     24
BI                            22
Cybersecurity Analyst         17
IoT Engineer                  13
Name: count, dtype: int64


3. Final Extraction

In [5]:
#output collectors
records = []
errors = []

#extraction loop
for i, (pdf_path, category) in enumerate(tqdm(all_files, desc="Extracting...")):
    result = extracted_resume_text(pdf_path)
    if result is None:
        text, err = "", "No extractable text"
    else:
        text, err = result
    if err:
        errors.append({ 'resume_id': i, 'file': pdf_path, 'category': category, 'error': err})

    #saving extracted texts
    records.append({
        'resume_id': i,
        'resume_text': text,
        'category': category,
        'file_path': pdf_path  #this path information is dropped in the final output
    })

#final save
pd.DataFrame(records).to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(records)} records to {OUTPUT_CSV}")

if errors:
    pd.DataFrame(errors).to_csv(ERROR_CSV, index=False)
    print(f"{len(errors)} files had extraction issues, see {ERROR_CSV}")

Extracting...:   1%|          | 15/1603 [00:02<02:59,  8.83it/s]Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Extracting...:   2%|▏         | 25/1603 [00:03<04:03,  6.48it/s]Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Extracting...:   2%|▏         | 33/1603 [00:05<04:59,  5.25it/s]Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Extracting...:   3%|▎         | 48/1603 [00:08<05:41,  4.55it/s]Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Extracting...:   4%|▎         | 57/1603 [00:09<03:23,  7

Saved 1603 records to ./resumes_extracted.csv
169 files had extraction issues, see ./resume_extraction_errors.csv
